In [4]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks


housing = fetch_california_housing(as_frame=True)
df = housing.data.copy()
df['MedHouseVal'] = housing.target

df['RoomsPerHousehold'] = df['AveRooms'] / df['AveOccup']
df['BedroomsPerRoom'] = df['AveBedrms'] / df['AveRooms']

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)


scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

input_layer = layers.Input(shape=(X_train_scaled.shape[1],))

deep = layers.Dense(128, activation='swish')(input_layer)
deep = layers.BatchNormalization()(deep)
deep = layers.Dropout(0.3)(deep)

deep = layers.Dense(64, activation='swish')(deep)
deep = layers.BatchNormalization()(deep)
deep = layers.Dropout(0.2)(deep)

deep = layers.Dense(32, activation='swish')(deep)


wide = layers.Dense(16, activation='linear')(input_layer)

merged = layers.concatenate([wide, deep])
output_layer = layers.Dense(1)(merged)

model = models.Model(inputs=input_layer, outputs=output_layer)


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss=tf.keras.losses.Huber(),
    metrics=['mean_absolute_error', 'mean_squared_error']
)

lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)


history = model.fit(
    X_train_scaled, y_train,
    epochs=150,
    batch_size=64,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)

test_metrics = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nOptimized Test MSE: {test_metrics[2]:.4f}")
print(f"Optimized Test MAE: {test_metrics[1]:.4f}")

Epoch 1/150
207/207 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.3018 - mean_absolute_error: 0.6116 - mean_squared_error: 1.6960 - val_loss: 0.2912 - val_mean_absolute_error: 0.5953 - val_mean_squared_error: 1.1716 - learning_rate: 0.0050
Epoch 2/150
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1968 - mean_absolute_error: 0.4751 - mean_squared_error: 2.1054 - val_loss: 0.2180 - val_mean_absolute_error: 0.5218 - val_mean_squared_error: 0.5159 - learning_rate: 0.0050
Epoch 3/150
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1792 - mean_absolute_error: 0.4461 - mean_squared_error: 2.0391 - val_loss: 0.2232 - val_mean_absolute_error: 0.4968 - val_mean_squared_error: 2.5846 - learning_rate: 0.0050
Epoch 4/150
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1688 - mean_absolute_error: 0.4317 - mean_squared_error: 0.5692 - val_loss: 0.2011 - val_mean_absolute_error: 0.4719 - val_mean_squared_error: 1.2362 - learning_rate: 0.0050
Epoch 5/150
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step